In [ ]:
# Read one parquet file only

from pyspark.sql import SparkSession
import pyarrow.parquet as pq

spark = SparkSession.builder.config("spark.sql.adaptive.enabled", "false").appName("parquet-inspect").getOrCreate()

path = "s3://ai-de-aws-learning-test/spark-learning/rides/tripdata/fhvhv_tripdata_2024-03.parquet" # file size is 450 MB

# --- Spark side ---
df = spark.read.parquet(path)

num_partitions = df.rdd.getNumPartitions()   # how Spark split the file for reading
total_records  = df.count()                  # full row count (triggers a scan)

# --- Parquet footer side (PyArrow) ---
pf = pq.ParquetFile(path)
num_row_groups   = pf.metadata.num_row_groups
rows_from_footer = pf.metadata.num_rows       # total rows straight from metadata (no scan)

print(f"Spark partitions : {num_partitions}")
print(f"Row groups       : {num_row_groups}")
print(f"Total records    : {total_records}")
print(f"Rows (footer)     : {rows_from_footer}")

for i in range(pf.metadata.num_row_groups):
    rg = pf.metadata.row_group(i)
    print(f"row group {i}: {rg.num_rows} rows, {rg.total_byte_size/1024/1024:.1f} MB")

In [ ]:
# path = "s3://ai-de-aws-learning-test/spark-learning/rides/tripdata/*.parquet"


# total 1376.4 mb of data, totalBytes = 1443260006 bytes
# available cpu cores = 8

# totalBytes/CPU_Cores = 1443260006/8 = 180407500 bytes = 172 mb

# maxPartitionBytes property is 128mb by default
# openCostInBytes default value is 4mb

# total partitions will be decided by maxSplitBytes = min(maxPartitionBytes, max(openCostInBytes , totalBytes/CPU_Cores) )

# maxSplitBytes = min ( 128mb , max(4mb , 172mb) ) = min ( 128mb , 172mb ) = 128mb

# partitions = totalBytes/maxSplitBytes = 1443260006/134217728 = 11 partitions


# you have 30 files, each one of it 1 mb = total size is 30 mb

# 1 files of 1 mb, when spark read it ( 1 mb + 4 mb ) = 5mb

# 30 * 5mb = 150 mb

# 150 mb (2 partitions)


In [ ]:
print("defaultParallelism  :", spark.sparkContext.defaultParallelism)
print("maxPartitionBytes   :", spark.conf.get("spark.sql.files.maxPartitionBytes"))
print("openCostInBytes     :", spark.conf.get("spark.sql.files.openCostInBytes"))

# These factors completely decides the number of partitions
# Actual formula is 

# maxSplitBytes = min( maxPartitionBytes,                 # 128 MB default
#                      max( openCostInBytes,              # 4 MB
#                           totalBytes / total executor cores ) )

# Total partitions = totalBytes / maxSplitBytes

# 1.) maxPartitionBytes - Maximum size of each partition

# 2.) openCostInBytes is a Spark configuration that tells Spark:
# “Assume opening a file has the same cost as reading this many bytes of data.”

# Why does Spark need this?

# Opening a file (especially on S3, ADLS, GCS, etc.) is not free. 
# Even if a file is only 10 KB, Spark still needs to:

# * Contact the storage service
# * Fetch metadata
# * Open a connection
# * Start reading

# This is mainly helpful for small files, For example

# You have 30 files, each 1 MB. Actual data size will be 30 MB
# Effective size Spark uses: 30 × (1 MB + 4 MB) = 150 MB because here Spark considering opening file of 1 MB size
# is equal to the efforts of opening 4 MB file

# Since one partition can hold only 128 MB, Spark creates 2 input partitions instead of 1.
# Without the open cost, Spark would have put all 30 files into a single partition.

# At first glance, yes, putting all 30 MB into one partition looks better because there is less scheduling overhead.

# But Spark isn’t optimizing only for data size—it’s optimizing for total work, which includes opening files.
# So having 2 partitions in this case will provide balanced parallelism


In [ ]:
# We can set partition size this way

spark.conf.set("spark.sql.files.maxPartitionBytes", str(64 * 1024 * 1024))  # 64 MB
df = spark.read.parquet(path)
print(df.rdd.getNumPartitions())   # ~8 now